# train_tabular_v79 — v75と同一構成、物理錨のみK16に差替
実行順: `create_candidates_v79.py` → `make_gf_v79.py` → 本ノート。
gf特徴はK16版(gf_delta=P6オフセット/gf_std=P6 postd/gf_vs_struct=K16σ ※列名は旧互換)。


# v47 tabular学習 — v28特徴一式 × PF7バンク[pf1/2/3 · r0/r1/r4 · pfA] = 全781列
v47/artifacts_v69/data/train.parquet(lag16, 7バンク)をスキーマ駆動で自動採用。pf_ancc/self/nbr_1..7・beam・formation(tvtF)・dense・v9 build_well 全部入り。LGBM×3+CatBoost×3+XGB×3 を GroupKFold(5)。順方向のみ。

In [2]:
# ===== v36: 順方向+逆方向DA ロード(pyarrow列チャンク→事前確保float32。RAM節約版) =====
import json, joblib, warnings, gc
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings("ignore")

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
ART = PROJECT / "v82" / "artifacts_v82a"
SRC_ART = PROJECT / "v79" / "artifacts_v79"   # 入力parquetはv79を流用
FWD_CANDS = [SRC_ART / "data" / "train.parquet"]
FWD_PQ = next((p for p in FWD_CANDS if p.exists()), FWD_CANDS[0])
BWD_PQ = SRC_ART / "data" / "train_backward.parquet"
USE_BACKWARD = False          # ★False=順方向のみ(ablation基準)。fold pklは自動で別フォルダ
MODELS = ART / ("models_bwd" if USE_BACKWARD else "models_fwd"); MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 1.0
pp_params = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}

USE_ORIGIN = {"base": True, "pf": True, "beam": True, "self": True, "nbr": True, "zgrad": True}
PREFIXES = [("self", "pf_self"), ("nbr", "pf_nbr"), ("beam", "beam_"), ("pf", "pf_")]
META = {"well", "id", "target", "is_backward"}
def origin_of(c):
    if c == "z_grad": return "zgrad"
    for o, pre in PREFIXES:
        if c.startswith(pre): return o
    return "base"

schema = pq.read_schema(FWD_PQ)
all_feats = [c for c in schema.names if c not in META]
features = [c for c in all_feats if USE_ORIGIN.get(origin_of(c), True)]
feat_all = features + (["is_backward"] if USE_BACKWARD else [])
print("origin別 全特徴数:", dict(Counter(origin_of(c) for c in all_feats)))
print("採用", len(feat_all), "列(is_backward含む)" if USE_BACKWARD else "列")
ART.joinpath("features.json").write_text(json.dumps(feat_all))
ART.joinpath("pp_params.json").write_text(json.dumps(pp_params))

N_FWD = pq.ParquetFile(FWD_PQ).metadata.num_rows
N_BWD = pq.ParquetFile(BWD_PQ).metadata.num_rows if USE_BACKWARD else 0
X = np.empty((N_FWD + N_BWD, len(feat_all)), np.float32)

def load_block(path, row0, feat_list):
    names = set(pq.read_schema(path).names)
    for j0 in range(0, len(feat_list), 24):
        sub = feat_list[j0:j0 + 24]
        have = [c for c in sub if c in names]
        tb = pq.read_table(path, columns=have) if have else None
        for jj, c in enumerate(sub):
            j = j0 + jj
            if c in names:
                X[row0:row0 + (N_BWD if row0 else N_FWD), j] = \
                    tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32, copy=False)
            else:
                X[row0:row0 + (N_BWD if row0 else N_FWD), j] = np.nan
        del tb
        if (j0 // 24) % 5 == 4: gc.collect()

load_block(FWD_PQ, 0, features)
if USE_BACKWARD:
    load_block(BWD_PQ, N_FWD, features)
    X[:N_FWD, len(features)] = 0.0; X[N_FWD:, len(features)] = 1.0

sf = pq.read_table(FWD_PQ, columns=["well", "target", "last_known_tvt", "md_since", "pf_ancc"])
g_f = sf.column("well").to_pandas().to_numpy()
y_f = sf.column("target").to_numpy(zero_copy_only=False).astype(np.float32)
lk  = sf.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds = sf.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = sf.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
del sf
if USE_BACKWARD:
    sb = pq.read_table(BWD_PQ, columns=["well", "target"])
    g = np.concatenate([g_f, sb.column("well").to_pandas().to_numpy()])
    y = np.concatenate([y_f, sb.column("target").to_numpy(zero_copy_only=False).astype(np.float32)])
    del sb
else:
    g, y = g_f, y_f
gc.collect()
print(f"rows: fwd {N_FWD} + bwd {N_BWD} = {len(X)} | features {len(feat_all)}"
      f" | wells {pd.unique(g).size} | X {X.nbytes/1e9:.1f} GB")

# ★v48 grid filter(PF不使用の物理予測, SIGG6/TAU0.35)を特徴に追加
_GFP = PROJECT / "v79" / "artifacts_v79" / "gf_feature_v79.npy"
if _GFP.exists():
    _GF = np.load(_GFP)[:N_FWD]
    _GF = np.nan_to_num(_GF, nan=0.0).astype(np.float32)
    _gcols = ["gf_delta", "gf_std", "gf_vs_struct"]
    X = np.hstack([X, _GF]); feat_all = feat_all + _gcols
    ART.joinpath("features.json").write_text(json.dumps(feat_all))
    print(f"★v79 K16-gf特徴 {len(_gcols)}列追加 → 計 {len(feat_all)}列")
else:
    print("[warn] gf_feature.npy 無し。先に python v79/make_gf_v79.py を実行")


# ★v82a: 素PF(v69_4=K16gate無しバンク)のpl_列を注入(行順一致検証済)
_SCP = PROJECT / "v82" / "artifacts_v82" / "v82_plainpf_train.parquet"
_sc = pq.read_table(_SCP)
_scw = _sc.column("well").to_pandas().to_numpy()
assert len(_scw) == N_FWD and (_scw == g_f).all(), "v82 plainpf行順不一致"
_sccols = [c for c in _sc.schema.names if c != "well"]
_SC = np.column_stack([_sc.column(c).to_numpy(zero_copy_only=False).astype(np.float32) for c in _sccols])
X = np.hstack([X, np.nan_to_num(_SC, nan=0.0)]); feat_all = feat_all + _sccols
ART.joinpath("features.json").write_text(json.dumps(feat_all))
del _sc, _SC
print(f"★v82a 素PF {len(_sccols)}列追加 → 計 {len(feat_all)}列")


origin別 全特徴数: {'base': 329, 'beam': 40, 'pf': 238, 'self': 28, 'nbr': 26, 'zgrad': 1}
採用 662 列
rows: fwd 3783989 + bwd 0 = 3783989 | features 662 | wells 773 | X 10.0 GB
★v48 gf特徴 3列追加 → 計 665列


In [3]:
# ===== モデル: LightGBM×3 + CatBoost×3 + XGBoost×3(v20系, GPU) =====
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f"lightgbm-{i+1}", "lgb", p) for i, p in enumerate(lgb_params)] +
            [(f"catboost-{i+1}", "cb", p) for i, p in enumerate(cb_params)] +
            [(f"xgboost-{i+1}", "xgb", p) for i, p in enumerate(xgb_params)])
print("variants:", [v[0] for v in VARIANTS])

variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [4]:
# ===== GroupKFold(5) 学習 → OOF + fold pkl保存(再開可) =====
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == "lgb":
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse", callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == "cb":
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m
gkf = GroupKFold(n_splits=N_SPLITS); folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        va = va[va < N_FWD]          # ★検証・OOFは順方向行のみ(逆行はtrain専用)
        fp = d / f"fold{k}.pkl"
        if fp.exists(): m = joblib.load(fp); tag = "load"
        else:
            m = fit_one(kind, params, X[tr], y[tr], X[va], y[va]); joblib.dump(m, fp); tag = "fit"
        oof[name][va] = m.predict(X[va])
        print(f"{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}", flush=True)
        del m; gc.collect()
    print(f"== {name} OOF rmse(順方向のみ)={root_mean_squared_error(y[:N_FWD], oof[name][:N_FWD]):.4f}")

# oof 6.87ぐらい

Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[556]	valid_0's rmse: 6.40435	valid_0's l2: 41.0157
lightgbm-1 fold0 [fit]: rmse=6.4043
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[343]	valid_0's rmse: 6.30618	valid_0's l2: 39.7679
lightgbm-1 fold1 [fit]: rmse=6.3062
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[510]	valid_0's rmse: 5.95906	valid_0's l2: 35.5104
lightgbm-1 fold2 [fit]: rmse=5.9591
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[131]	valid_0's rmse: 6.25883	valid_0's l2: 39.173
lightgbm-1 fold3 [fit]: rmse=6.2588
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[287]	valid_0's rmse: 6.45506	valid_0's l2: 41.6678
lightgbm-1 fold4 [fit]: rmse=6.4551
== lightgbm-1 OOF rmse(順方向のみ)=6.2791
Training until validation scores don't 

In [5]:
# ===== アンサンブル OOF + apply_pp(submit準拠: pf_res=pf_ancc-lk) → overall RMSE =====
names = [v[0] for v in VARIANTS]
oof_ens = np.mean([oof[n] for n in names], axis=0)[:N_FWD]
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85); alpha = float(pp_params.get("alpha", 1.0))
pf_res = pf_abs - lk
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
raw = root_mean_squared_error(y[:N_FWD], oof_ens); ppr = root_mean_squared_error(y[:N_FWD], apply_pp_res(oof_ens))
print("=== OOF overall RMSE(全行プール=Kaggle指標) ===")
print(f"  raw(GBMアンサンブル) : {raw:.4f}")
print(f"  + apply_pp           : {ppr:.4f}")
print("  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。")

=== OOF overall RMSE(全行プール=Kaggle指標) ===
  raw(GBMアンサンブル) : 6.2335
  + apply_pp           : 6.2157
  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。


In [6]:
# ===== v29 NN(combo) との残差ブレンド + 公平基準(旧v28 GBM×NN を同一行で再計算) =====
V29_ART  = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29")
V29B_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29b")
_step = max(1, int(round(1.0 / ROW_FRAC))) if ROW_FRAC < 0.6 else 1
nnz = np.load(V29B_ART / "nn_oof_combo.npz")
nn_ens = np.mean([nnz[k] for k in nnz.files], axis=0)[::_step]
gbm_old = np.load(V29_ART / "gbm_oof.npy")[::_step]
assert len(nn_ens) == N_FWD == len(gbm_old), (len(nn_ens), N_FWD, len(gbm_old))
def overall_pp(res): return root_mean_squared_error(y[:N_FWD], apply_pp_res(res))
ws2 = np.arange(0.0, 1.001, 0.02)
scs0 = [overall_pp((1 - w2) * gbm_old + w2 * nn_ens) for w2 in ws2]
b0 = int(np.argmin(scs0))
scs = [overall_pp((1 - w2) * oof_ens + w2 * nn_ens) for w2 in ws2]
bi = int(np.argmin(scs))
print(f"公平基準(旧v28GBM×NN, 同一行): 旧GBM単体 {overall_pp(gbm_old):.4f} / blend w_nn={ws2[b0]:.2f} -> {scs0[b0]:.4f}")
print(f"GBM(v36)単体+pp             : {overall_pp(oof_ens):.4f}")
print(f"blend GBM(v36)×NN           : w_nn={ws2[bi]:.2f} -> {scs[bi]:.4f}  (Δ vs 基準 {scs[bi] - scs0[b0]:+.4f})")
print("※ 行はROW_FRAC間引き後の同一行同士。全行の絶対値(6.8711等)とは僅かにずれるがΔは公平。")
np.save(ART / "gbm_v41_oof.npy", oof_ens)
print("saved:", ART / "gbm_v41_oof.npy")


公平基準(旧v28GBM×NN, 同一行): 旧GBM単体 6.9010 / blend w_nn=0.60 -> 6.7322
GBM(v36)単体+pp             : 6.2157
blend GBM(v36)×NN           : w_nn=0.24 -> 6.1522  (Δ vs 基準 -0.5801)
※ 行はROW_FRAC間引き後の同一行同士。全行の絶対値(6.8711等)とは僅かにずれるがΔは公平。
saved: C:\Users\kosaka256\Documents\rogii_claude\v75\artifacts_v75\gbm_v41_oof.npy
